# Pipeline modules — interactive test notebook (small-LLM deployment chain)

This is the **focused deployment chain** for the Yorùbá voice-query pipeline. No RAG, no FAISS, no Mistral-7B — the M4 stage is a small instruct-tuned LLM (Qwen2.5-1.5B-Instruct) that answers the English query directly. The whole chain stays fast, portable, and easy to debug.

```
audio.wav                                                              ┌─ small LLM answers
  └─► M1 ASR      (whisper-large-v3 fine-tune, mlx or hf)          ───►│   directly
        └─► M2 Diacritic   (Davlan/mT5_base_yoruba_adr)                │
              └─► M3 Translate (NLLB-200, yor_Latn → eng_Latn) ────────┘
                                                                       │
                                                                       ▼
                                                        EN answer
                                                                       │
                                                                       ▼
                                              ┌── M5 TTS (NLLB en→yo + facebook/mms-tts-yor) ──┐
                                              │                                                │
                                              └─► response.wav ────────────────────────────────┘
```

**Why this layout (and not the M4_RAG / M4_Chat versions in `pipeline.py`)**:

- **No knowledge retrieval needed.** The small LLM has enough world knowledge for the kind of factual Yorùbá queries this pipeline targets ("Who is X?", "What is Y?"). Wikipedia-RAG was overkill.
- **No 4 GB GGUF.** Qwen2.5-1.5B-Instruct loads in seconds, runs on any device, doesn't need `llama-cpp-python` or a built FAISS index.
- **Fewer moving parts.** Five module loads instead of five + FAISS + chunking + retrieval scoring. Easier to debug, easier to ship.

This notebook **is** the deployment surface. The per-module sections (M1, M2, M3, M4, M5) are for poking at one stage when something's wrong. The chain section at the bottom runs the whole thing end-to-end.

**Heads up — slow first calls**: `initialize()` on M1/M2/M3/M5 downloads HF weights the first time (~6 GB total across modules). Subsequent calls reuse the loaded models.

For batch evaluation (YASR-Bench WER), see `Whisper_test.ipynb`. For fine-tuning, see `Whisper_v4.ipynb`.

## Setup

Run this cell once. Picks up `config.py`, makes sure `data/audio/fleurs_yo_sample.wav` exists (so M1 has something to chew on), and configures logging the same way the CLI does.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
assert (ROOT / "pipeline.py").exists(), f"Run this notebook from the repo root (got {ROOT})."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
from utils.logging import get_logger, log_stage

log = get_logger("notebook")

SAMPLE_WAV = config.AUDIO_DIR / "fleurs_yo_sample.wav"
if not SAMPLE_WAV.exists():
    print(f"Fetching {SAMPLE_WAV} (one-time)…")
    import subprocess
    subprocess.run([sys.executable, "tests/fetch_yoruba_sample.py"], check=True)

print(f"sample audio  → {SAMPLE_WAV}")
print(f"logs dir      → {config.LOG_DIR}")
print(f"M1 mlx model  → {config.M1_MODEL}")
print(f"M1 hf  model  → {config.M1_HF_MODEL}")

## M1 — ASR (audio → raw Yorùbá)

Two backends share the `M1ASR` / `M1ASRHF` API:

- **`mlx`** — `M1ASR`, mlx-whisper, fast on Apple Silicon. Default in `pipeline.py`.
- **`hf`** — `M1ASRHF`, loads from a HF repo ID **or a local path**. Uses `config.M1_HF_MODEL` by default (currently `whisper-large-v3-yoruba-v3`); override with `M1_HF_MODEL_OVERRIDE` below to test a different checkpoint without editing `config.py`.

**Common overrides**:
- A local path to a merged 16-bit checkpoint downloaded from Drive (e.g. the v4 fine-tune).
- A different HF Hub repo ID (e.g. `devalade/whisper-large-v3-yoruba-v4` once pushed).
- The base model (`openai/whisper-large-v3`) to A/B against your fine-tune live.

Audio plays inline so you can hear what's being transcribed.

In [ ]:
from IPython.display import Audio, display

M1_BACKEND = "mlx"   # "mlx" or "hf"
M1_INPUT   = SAMPLE_WAV   # or any other Path to a wav/mp3

# Override the HF model. None = use config.M1_HF_MODEL (currently v3 on HF Hub).
# Examples (uncomment one to test):
#   M1_HF_MODEL_OVERRIDE = "/Users/macuser/models/whisper-yo-v4-merged"
#   M1_HF_MODEL_OVERRIDE = "devalade/whisper-large-v3-yoruba-v4"
#   M1_HF_MODEL_OVERRIDE = "openai/whisper-large-v3"   # baseline A/B
M1_HF_MODEL_OVERRIDE = None

if M1_BACKEND == "hf":
    from modules.m1_asr_hf import M1ASRHF as _M1
    if M1_HF_MODEL_OVERRIDE:
        m1 = _M1(model=M1_HF_MODEL_OVERRIDE)
        m1_label = f"hf ({M1_HF_MODEL_OVERRIDE})"
    else:
        m1 = _M1()
        m1_label = f"hf ({config.M1_HF_MODEL})"
else:
    from modules.m1_asr import M1ASR as _M1
    m1 = _M1()
    m1_label = f"mlx ({config.M1_MODEL})"

m1.initialize()
r1 = m1.process(str(M1_INPUT))
log_stage("M1", f"nb-{Path(M1_INPUT).stem}", r1)

print(f"backend  : {m1_label}")
print(f"transcript:\n  {r1['text']}")
print(f"segments : {len(r1.get('segments', []))}")
display(Audio(str(M1_INPUT)))

## M2 — Diacritic restoration (raw Yorùbá → diacritized Yorùbá)

Davlan/mT5_base_yoruba_adr. Restores tone marks and sub-dotted letters on undiacritized text. The current fine-tune outputs diacritics directly from M1, so M2 is mostly a safety net — but worth testing in isolation when iterating on the model.

In [ ]:
from modules.m2_diacritic import M2Diacritic

# Edit this to test specific inputs. Default uses M1 output if available.
M2_INPUT = r1["text"] if "r1" in dir() else "bawo ni o se wa loni"

m2 = M2Diacritic()
m2.initialize()
r2 = m2.process(M2_INPUT)
log_stage("M2", "nb-m2", {"input": M2_INPUT, **r2})

print(f"in  : {M2_INPUT}")
print(f"out : {r2['text']}")

## M3 — Translation YO → EN

NLLB-200 distilled-600M, `yor_Latn` → `eng_Latn`. Takes the diacritized Yorùbá from M2 and produces the English query that goes into M4.

In [ ]:
from modules.m3_translate import M3Translate

M3_INPUT = r2["text"] if "r2" in dir() else "báwo ni ó ṣe wá lónìí"

m3 = M3Translate()
m3.initialize()
r3 = m3.process(M3_INPUT)
log_stage("M3", "nb-m3", r3)

print(f"YO in  : {M3_INPUT}")
print(f"EN out : {r3['text']}")

## M4 — small-model chat (skips RAG)

Default `M4RAG` needs FAISS + the Wikipedia index, and `M4Chat` loads Mistral-7B (4 GB GGUF). Both overkill for ad-hoc testing of the chain.

This swaps in **`Qwen/Qwen2.5-1.5B-Instruct`** (~3 GB in bf16 / ~1 GB int4) — small enough to spin up in any notebook, capable enough to answer general factual questions about Yorùbá topics that come out of M3. Output shape matches `M4RAG`/`M4Chat` (`{"answer", "max_sim", "below_threshold"}`) so M5 chains in unchanged.

To go back to the production M4, import `M4RAG` or `M4Chat` from `modules/`. This notebook intentionally avoids both for the lightweight-iteration flow.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SMALL_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

class M4Small:
    """Drop-in replacement for M4RAG / M4Chat using a small instruct model.
    No retrieval, no GGUF, no FAISS — just a chat model loaded straight from HF.
    Same return shape as the production modules so M5 chains in without changes."""

    name = "M4Small"

    def __init__(self, model_id: str = SMALL_MODEL_ID):
        self.model_id = model_id
        self.tok = None
        self.mdl = None
        self._ready = False

    def initialize(self) -> None:
        self.tok = AutoTokenizer.from_pretrained(self.model_id)
        self.mdl = AutoModelForCausalLM.from_pretrained(
            self.model_id, torch_dtype="auto", device_map="auto",
        ).eval()
        self._ready = True

    def process(self, text: str) -> dict:
        messages = [
            {"role": "system",
             "content": "You are a concise assistant. Answer in 2-3 sentences."},
            {"role": "user", "content": text},
        ]
        prompt = self.tok.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt",
        ).to(self.mdl.device)
        with torch.inference_mode():
            out = self.mdl.generate(
                prompt, max_new_tokens=200, do_sample=False,
                pad_token_id=self.tok.eos_token_id,
            )
        answer = self.tok.decode(out[0, prompt.shape[-1]:], skip_special_tokens=True).strip()
        return {"answer": answer, "max_sim": float("nan"), "below_threshold": False}


M4_INPUT = r3["text"] if "r3" in dir() else "Who is Wole Soyinka?"

m4 = M4Small()
m4.initialize()
r4 = m4.process(M4_INPUT)
log_stage("M4", "nb-m4-small", r4)

print(f"model: {SMALL_MODEL_ID}")
print(f"Q    : {M4_INPUT}")
print(f"A    : {r4['answer']}")

## M5 — TTS (English answer → Yorùbá speech)

Two-leg: NLLB-200 en→yo, then `facebook/mms-tts-yor` synthesises the diacritized Yorùbá. Output WAV plays inline.

In [ ]:
from modules.m5_tts import M5TTS

M5_INPUT  = r4["answer"] if "r4" in dir() else "Lagos is the largest city in Nigeria."
M5_OUTPUT = config.OUTPUT_DIR / "nb_m5_test.wav"
M5_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

m5 = M5TTS()
m5.initialize()
r5 = m5.process(M5_INPUT, output_path=str(M5_OUTPUT))
log_stage("M5", "nb-m5", r5)

print(f"EN in       : {r5['en']}")
print(f"YO (raw)    : {r5['yo']}")
print(f"YO (diacrit): {r5['yo_diacritized']}")
print(f"WAV         : {r5['audio_path']}  ({r5['duration_s']:.2f}s @ {r5['sampling_rate']} Hz)")
display(Audio(r5["audio_path"]))

## Full deployment chain — M1 → M2 → M3 → M4Small → M5

**This is the chain.** Assembles M1→M5 manually with the small-LLM M4 (defined in the M4 section above). No `YorubaPipeline`, no FAISS, no Mistral — what you ship is exactly what runs here.

Reuses modules already loaded by the per-section cells if you ran them; spins up any that aren't. Honors `M1_BACKEND` and `M1_HF_MODEL_OVERRIDE` from the M1 cell, so flipping between mlx, the production v3 HF model, your v4 checkpoint, or the baseline takes one variable change.

If you want the **production CLI chain** that matches `make run` / `pipeline.py` exactly (M4 = RAG with Mistral-7B), call `from pipeline import YorubaPipeline` in a fresh cell — but that's the heavier alternative we explicitly moved away from for this deployment.

In [ ]:
from modules.m1_asr import M1ASR
from modules.m1_asr_hf import M1ASRHF
from modules.m2_diacritic import M2Diacritic
from modules.m3_translate import M3Translate
from modules.m5_tts import M5TTS

# Inherits M1_BACKEND and M1_HF_MODEL_OVERRIDE from the M1 cell. To pin here:
ASR        = M1_BACKEND if "M1_BACKEND" in dir() else "mlx"
HF_OVERRIDE = M1_HF_MODEL_OVERRIDE if "M1_HF_MODEL_OVERRIDE" in dir() else None

CHAIN_IN   = SAMPLE_WAV
CHAIN_OUT  = config.OUTPUT_DIR / "nb_chain_small.wav"
CHAIN_OUT.parent.mkdir(parents=True, exist_ok=True)

def _get_or_init(varname, ctor):
    """Reuse a module if it's already loaded in the notebook's namespace,
    otherwise construct + initialize a fresh one. Saves a re-download per cell."""
    g = globals()
    obj = g.get(varname)
    if obj is None:
        obj = ctor(); obj.initialize(); g[varname] = obj
    return obj

if ASR == "hf":
    m1_ctor = (lambda: M1ASRHF(model=HF_OVERRIDE)) if HF_OVERRIDE else M1ASRHF
else:
    m1_ctor = M1ASR

m1 = _get_or_init("m1", m1_ctor)
m2 = _get_or_init("m2", M2Diacritic)
m3 = _get_or_init("m3", M3Translate)
m4 = _get_or_init("m4", M4Small)             # small-LLM M4 defined above
m5 = _get_or_init("m5", M5TTS)

run_id = f"nb-chain-small-{Path(CHAIN_IN).stem}"
c1 = m1.process(str(CHAIN_IN));                 log_stage("M1", run_id, c1)
c2 = m2.process(c1["text"]);                    log_stage("M2", run_id, c2)
c3 = m3.process(c2["text"]);                    log_stage("M3", run_id, c3)
c4 = m4.process(c3["text"]);                    log_stage("M4", run_id, c4)
c5 = m5.process(c4["answer"], output_path=str(CHAIN_OUT))
log_stage("M5", run_id, c5)

print(f"\nM1 backend: {ASR}" + (f"  override={HF_OVERRIDE}" if HF_OVERRIDE else ""))
print("\n=== M1 raw YO     ===\n", c1["text"])
print("\n=== M2 diacrit.   ===\n", c2["text"])
print("\n=== M3 EN query   ===\n", c3["text"])
print("\n=== M4 EN answer  ===\n", c4["answer"])
print("\n=== M5 YO answer  ===\n", c5["yo_diacritized"])
print(f"\nWAV: {c5['audio_path']}  ({c5['duration_s']:.2f}s)")
print(f"log: logs/{run_id}.jsonl")

display(Audio(c5["audio_path"]))

## Where things land

- **Per-stage JSONL logs**: `logs/<run-id>.jsonl` — same format `pipeline.py` writes. Useful for comparing notebook runs against CLI runs.
- **Audio outputs**: `data/outputs/nb_*.wav` (this notebook) and `data/outputs/response.wav` (CLI). They live in the same directory so `make clean-outputs` clears both.
- **No new tracking systems here**: WER / YASR-Bench logging lives in `Whisper_test.ipynb` and writes to Drive. This notebook is for *function* testing, not metric tracking.